# 🏎️ F1 Podium Prediction System — 2026 Season
### Linear Algebra Pipeline · UE24MA241B · PES University
**Predicts the next 3 upcoming races using weighted least squares trained on 2025 + 2026 data**

---
```
Real-World Data (FastF1)
      ↓
Step 1 : Matrix Representation   — Driver × Feature matrix A, target b
Step 2 : Matrix Simplification   — RREF / Gaussian Elimination
Step 3 : Structure of Space      — Column Space & Null Space (SVD)
Step 4 : Remove Redundancy       — Basis Selection via correlation
Step 5 : Orthogonalization       — Gram-Schmidt / QR Decomposition
Step 6 : Projection + Least Sq.  — Weighted: x̂ = (AᵀWA)⁻¹AᵀWb
Step 7 : Pattern Discovery       — Eigenvalues of AᵀWA
Step 8 : Final Output            — Podium predictions for next 3 races
```

---
## ⚙️ Cell 1 — Setup
Run once per session. Installs FastF1, mounts Drive, silences logs.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — SETUP
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys
def install(p): subprocess.check_call([sys.executable,"-m","pip","install",p,"-q"])
print("📦 Installing dependencies...")
install("fastf1"); install("ipywidgets")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import warnings, os, logging

# ── Silence FastF1 logs ───────────────────────────────────────────
import logging
# Suppress ALL fastf1 output completely
for lg in logging.Logger.manager.loggerDict:
    if "fastf1" in lg or "urllib" in lg or "requests" in lg:
        logging.getLogger(lg).setLevel(logging.CRITICAL)
        logging.getLogger(lg).propagate = False
for lg in ["fastf1","fastf1.core","fastf1.req","fastf1._api",
           "fastf1.ergast","fastf1.livetiming","urllib3","requests"]:
    l = logging.getLogger(lg)
    l.setLevel(logging.CRITICAL)
    l.propagate = False
    l.handlers = []
warnings.filterwarnings("ignore")

# ── Google Drive cache ────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CACHE_DIR = "/content/drive/MyDrive/f1_cache"
    IN_COLAB  = True
    print("✅ Google Drive mounted")
except Exception:
    CACHE_DIR = "./f1_cache"
    IN_COLAB  = False
    print("ℹ️  Local mode — cache: ./f1_cache")

os.makedirs(CACHE_DIR, exist_ok=True)

try:
    import fastf1
    fastf1.Cache.enable_cache(CACHE_DIR)
    USE_REAL = True
    print("✅ FastF1 ready — real data will be used")
except Exception as e:
    USE_REAL = False
    print(f"⚠️  FastF1 unavailable — synthetic fallback active")

# ── Colour constants ─────────────────────────────────────────────
F1_RED   = "#E10600"
F1_GOLD  = "#FFD700"
F1_WHITE = "#FFFFFF"
TEAM_COLORS = {
    "Red Bull Racing":"#3671C6","Mercedes":"#27F4D2","Ferrari":"#E8002D",
    "McLaren":"#FF8000","Aston Martin":"#229971","Williams":"#64C4FF",
    "Alpine":"#0093CC","Haas F1 Team":"#B6BABD","RB":"#6692FF",
    "Kick Sauber":"#52E252","Audi":"#FF5F00","Cadillac":"#B01E23",
}

# ── Plot style ───────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":"#0f0f0f","axes.facecolor":"#1a1a1a",
    "axes.edgecolor":"#333","axes.labelcolor":"#ccc",
    "xtick.color":"#888","ytick.color":"#888","text.color":"#ddd",
    "grid.color":"#2a2a2a","grid.linestyle":"--",
    "font.family":"monospace","axes.titlesize":12,"axes.labelsize":10,
})
print("\n🏎️  Setup complete — run all remaining cells then click Run Pipeline")


---
## 🎛️ Cell 2 — Control Panel

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — INTERFACE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import ipywidgets as widgets
from IPython.display import display, clear_output

# Fixed settings — no dropdowns needed
PREDICT_NEXT = 1      # always predict next race only
TOP_N        = 3      # always top 3 drivers

style  = {"description_width": "170px"}
layout = widgets.Layout(width="440px")

w_train_2025 = widgets.IntSlider(
    value=20, min=5, max=24, step=1,
    description="📚 2025 rounds to use:",
    style=style, layout=layout, continuous_update=False)

w_btn    = widgets.Button(
    description="  ▶  Predict Next Race",
    button_style="danger",
    layout=widgets.Layout(width="240px", height="42px"))

w_status = widgets.HTML(
    value="<i style='color:#888'>Ready — click Predict Next Race after all cells are loaded</i>")

info_html = """
<div style='background:#1a1a1a;border:1px solid #333;border-radius:6px;
padding:12px;font-family:monospace;font-size:12px;color:#aaa;margin-top:8px'>
<b style='color:#FFD700'>Predicting:</b>  Next race — Top 3 podium finishers<br>
<b style='color:#E10600'>Training:</b>  2025 full season + 2026 completed races<br>
<b style='color:#27F4D2'>Weighting:</b>  2026 → 3.0×  |  late 2025 → 1.5×  |  early 2025 → 0.5×<br>
<b style='color:#aaa'>Step 6:</b>  Weighted Least Squares  x̂ = (AᵀWA)⁻¹AᵀWb
</div>"""

box = widgets.VBox([
    widgets.HTML("<h3 style='color:#E10600;font-family:monospace;margin:0'>🏎️  F1 2026 — Next Race Podium Predictor</h3>"),
    w_train_2025,
    widgets.HBox([w_btn, widgets.Label("  ← click to predict")]),
    w_status,
    widgets.HTML(info_html),
], layout=widgets.Layout(border="1px solid #333", padding="18px", border_radius="8px"))
display(box)

def on_run(b):
    w_status.value = "<span style='color:#FFD700'>⏳ Running pipeline...</span>"
    w_btn.disabled = True
    try:
        run_full_pipeline(
            n_train_2025 = w_train_2025.value,
            top_n        = TOP_N,
            predict_next = PREDICT_NEXT,
        )
    except Exception as err:
        import traceback
        w_status.value = f"<span style='color:#E10600'>❌ {err}</span>"
        traceback.print_exc()
    finally:
        w_btn.disabled = False

w_btn.on_click(on_run)
print("✅ Control panel ready")


---
## 📡 Cell 3 — Data Loader
FastF1 → Feature Matrix A with row weights W.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — DATA LOADER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
F1_RED='#E10600'; F1_GOLD='#FFD700'; F1_WHITE='#FFFFFF'
TEAM_COLORS={"Red Bull Racing":"#3671C6","Mercedes":"#27F4D2","Ferrari":"#E8002D",
    "McLaren":"#FF8000","Aston Martin":"#229971","Williams":"#64C4FF",
    "Alpine":"#0093CC","Haas F1 Team":"#B6BABD","RB":"#6692FF",
    "Kick Sauber":"#52E252","Audi":"#FF5F00","Cadillac":"#B01E23"}

FEATURE_NAMES = ["Quali Gap (s)","Tire Factor","Env. Index","Mean/FP2 Lap (s)","Pitstop (s)"]
COMPOUND_MAP  = {"SOFT":1.0,"MEDIUM":2.0,"HARD":3.0,"INTERMEDIATE":2.5,"WET":3.5}

# Historical avg pitstop by team (used when race data unavailable)
HIST_PIT = {"Red Bull Racing":23.1,"Mercedes":23.8,"Ferrari":23.5,"McLaren":23.4,
            "Aston Martin":24.6,"Williams":25.1,"Alpine":24.9,"Haas F1 Team":25.3,
            "RB":24.8,"Kick Sauber":25.0,"Audi":25.5,"Cadillac":26.0}

def safe_f(v, default):
    try:
        r = float(v)
        return default if r!=r else r
    except: return default

# ── Synthetic data for fallback (2025 season approximation) ──────
import numpy as _np
_SYNTH_2025 = {}
_base_drivers = [
    ("NOR","McLaren"),("VER","Red Bull Racing"),("LEC","Ferrari"),
    ("PIA","McLaren"),("RUS","Mercedes"),("SAI","Ferrari"),
    ("HAM","Mercedes"),("ALO","Aston Martin"),("ANT","Mercedes"),("ALB","Williams")
]
_np.random.seed(42)
for rnd in range(1,25):
    rows=[]
    for pos,(abbr,team) in enumerate(_base_drivers[:10],1):
        rows.append((abbr,team,
            round(pos*0.08+_np.random.uniform(0,0.05),3),
            round(2.0+_np.random.uniform(0,1.5),3),
            round(0.63+_np.random.uniform(0,0.1),3),
            round(91+pos*0.2+_np.random.uniform(-0.3,0.3),3),
            round(23+pos*0.15+_np.random.uniform(0,0.5),2),
            pos))
    _SYNTH_2025[rnd]=rows

_SYNTH_2026 = {}
_base_2026 = [
    ("RUS","Mercedes"),("ANT","Mercedes"),("LEC","Ferrari"),
    ("HAM","Ferrari"),("NOR","McLaren"),("PIA","McLaren"),
    ("VER","Red Bull Racing"),("ALB","Williams"),("HUL","Audi"),("DOO","Cadillac")
]
for rnd in range(1,6):
    rows=[]
    for pos,(abbr,team) in enumerate(_base_2026[:10],1):
        rows.append((abbr,team,
            round(pos*0.07+_np.random.uniform(0,0.04),3),
            round(2.0+_np.random.uniform(0,1.2),3),
            round(0.65+_np.random.uniform(0,0.08),3),
            round(90.5+pos*0.18+_np.random.uniform(-0.2,0.2),3),
            round(23+pos*0.12+_np.random.uniform(0,0.4),2),
            pos))
    _SYNTH_2026[rnd]=rows

def _synth_round(season, rnd, top_n):
    src = _SYNTH_2026 if season==2026 else _SYNTH_2025
    rows = src.get(rnd, list(src.values())[0])[:top_n]
    drivers = [r[0] for r in rows]; teams=[r[1] for r in rows]
    feats=_np.array([[r[2],r[3],r[4],r[5],r[6]] for r in rows])
    pos  =_np.array([float(r[7]) for r in rows])
    return drivers,teams,feats,pos

def _suppress_ff1_logs():
    import logging
    for lg in ["fastf1","fastf1.core","fastf1.req","fastf1._api",
               "fastf1.ergast","urllib3","requests"]:
        l = logging.getLogger(lg)
        l.setLevel(logging.CRITICAL)
        l.propagate = False

def _load_real_round(season, rnd, top_n, future=False):
    """Load one race session from FastF1. future=True → use FP2 for lap times."""
    _suppress_ff1_logs()
    try:
        sess = fastf1.get_session(season, rnd, "R")
        sess.load(telemetry=False, weather=True, messages=False)
        quali = fastf1.get_session(season, rnd, "Q")
        quali.load(telemetry=False)

        res = sess.results.sort_values("Position").dropna(subset=["Position"]).head(top_n)
        drivers  = res["Abbreviation"].tolist()
        teams    = res["TeamName"].tolist()
        positions= res["Position"].astype(float).tolist()

        # C1 — Quali gap
        q_res = quali.results.set_index("Abbreviation")
        best_t = None
        for qc in ["Q3","Q2","Q1"]:
            if qc in q_res.columns:
                ts = q_res[qc].dropna()
                if not ts.empty:
                    best_t = ts.dt.total_seconds().min(); break
        best_t = best_t or 90.0
        q_gaps=[]
        for d in drivers:
            t=None
            for qc in ["Q3","Q2","Q1"]:
                if qc in q_res.columns and d in q_res.index:
                    raw=q_res.loc[d,qc]
                    if pd.notna(raw):
                        t=safe_f(raw.total_seconds() if hasattr(raw,"total_seconds") else raw, None)
                        if t: break
            q_gaps.append(round((t-best_t) if t else float(_np.nanmean([x for x in q_gaps if x==x]) if q_gaps else 1.5),4))

        # C2 — Tire factor
        tire_f=[]
        for d in drivers:
            try:
                laps=sess.laps.pick_drivers(d)
                if laps.empty: tire_f.append(2.0); continue
                last=laps.iloc[-1]
                cmp=COMPOUND_MAP.get(str(last["Compound"]).upper(),2.0)
                age=safe_f(last["TyreLife"],20)
                tire_f.append(round(cmp*(1+age/60),3))
            except: tire_f.append(2.0)

        # C3 — Env index
        try:
            wx=sess.weather_data
            tt=safe_f(wx["TrackTemp"].mean(),38.0) if not wx.empty else 38.0
            rain=1.0 if (not wx.empty and bool(wx["Rainfall"].any())) else 0.0
            env_v=round(tt/60+rain,3)
        except: env_v=0.65
        env=[env_v]*len(drivers)

        # C4 — Mean lap (race) or FP2 pace
        mean_l=[]
        for d in drivers:
            try:
                laps=sess.laps.pick_drivers(d).pick_quicklaps()
                if laps.empty: mean_l.append(93.0); continue
                lt=laps["LapTime"].dt.total_seconds().dropna()
                mean_l.append(round(float(lt.mean()),3) if not lt.empty else 93.0)
            except: mean_l.append(93.0)

        # C5 — Pitstop
        pit_t=[]
        for d,team in zip(drivers,teams):
            try:
                laps=sess.laps.pick_drivers(d)
                pi=laps["PitInTime"].dropna(); po=laps["PitOutTime"].dropna()
                if pi.empty or po.empty: pit_t.append(HIST_PIT.get(team,25.0)); continue
                n=min(len(pi),len(po))
                dur=[(po.iloc[i]-pi.iloc[i]).total_seconds() for i in range(n)]
                dur=[x for x in dur if 15<x<60]
                pit_t.append(round(float(_np.mean(dur)),2) if dur else HIST_PIT.get(team,25.0))
            except: pit_t.append(HIST_PIT.get(team,25.0))

        feats=_np.array([q_gaps,tire_f,env,mean_l,pit_t],dtype=float).T
        # NaN guard
        for col in range(feats.shape[1]):
            m=_np.isnan(feats[:,col])
            if m.any():
                med=float(_np.nanmedian(feats[:,col])) if not _np.all(m) else 0.5
                feats[m,col]=med
        return drivers,teams,feats,_np.array(positions)
    except Exception as ex:
        print(f"  ⚠️  FastF1 failed R{rnd}/{season}: {ex} — synthetic")
        return _synth_round(season,rnd,top_n)

def _load_future_quali(season, rnd, top_n):
    """For an upcoming race: load quali + FP2 instead of race data."""
    _suppress_ff1_logs()
    try:
        quali=fastf1.get_session(season,rnd,"Q")
        quali.load(telemetry=False)
        fp2=fastf1.get_session(season,rnd,"FP2")
        fp2.load(telemetry=False,weather=True)

        res=quali.results.sort_values("Q3").head(top_n)
        # fallback to overall classification if Q3 empty
        if res.empty or res["Q3"].isna().all():
            res=quali.results.head(top_n)
        drivers=res["Abbreviation"].tolist()
        teams  =res["TeamName"].tolist()

        q_res=quali.results.set_index("Abbreviation")
        best_t=None
        for qc in ["Q3","Q2","Q1"]:
            if qc in q_res.columns:
                ts=q_res[qc].dropna()
                if not ts.empty: best_t=ts.dt.total_seconds().min(); break
        best_t=best_t or 90.0
        q_gaps=[]
        for d in drivers:
            t=None
            for qc in ["Q3","Q2","Q1"]:
                if qc in q_res.columns and d in q_res.index:
                    raw=q_res.loc[d,qc]
                    if pd.notna(raw):
                        t=safe_f(raw.total_seconds() if hasattr(raw,"total_seconds") else raw,None)
                        if t: break
            q_gaps.append(round((t-best_t) if t else 1.5,4))

        # tire factor from FP2 last stint
        tire_f=[]
        for d in drivers:
            try:
                laps=fp2.laps.pick_drivers(d)
                if laps.empty: tire_f.append(2.0); continue
                last=laps.iloc[-1]
                cmp=COMPOUND_MAP.get(str(last["Compound"]).upper(),2.0)
                age=safe_f(last["TyreLife"],15)
                tire_f.append(round(cmp*(1+age/60),3))
            except: tire_f.append(2.0)

        # env from FP2 weather
        try:
            wx=fp2.weather_data
            tt=safe_f(wx["TrackTemp"].mean(),38.0) if not wx.empty else 38.0
            rain=1.0 if (not wx.empty and bool(wx["Rainfall"].any())) else 0.0
            env_v=round(tt/60+rain,3)
        except: env_v=0.65
        env=[env_v]*len(drivers)

        # C4 — FP2 representative pace (median of quicklaps)
        fp2_pace=[]
        for d in drivers:
            try:
                laps=fp2.laps.pick_drivers(d).pick_quicklaps()
                if laps.empty: fp2_pace.append(93.0); continue
                lt=laps["LapTime"].dt.total_seconds().dropna()
                fp2_pace.append(round(float(lt.median()),3) if not lt.empty else 93.0)
            except: fp2_pace.append(93.0)

        # C5 — historical pit average for team
        pit_t=[HIST_PIT.get(t,25.0) for t in teams]

        feats=_np.array([q_gaps,tire_f,env,fp2_pace,pit_t],dtype=float).T
        for col in range(feats.shape[1]):
            m=_np.isnan(feats[:,col])
            if m.any():
                med=float(_np.nanmedian(feats[:,col])) if not _np.all(m) else 0.5
                feats[m,col]=med
        # no actual positions for future race
        return drivers,teams,feats,None
    except Exception as ex:
        print(f"  ⚠️  Future quali load failed R{rnd}/{season}: {ex} — synthetic")
        drv,teams_s,feats_s,_=_synth_round(season,rnd,top_n)
        return drv,teams_s,feats_s,None

def build_training_matrix(n_2025, top_n, season_2026_done):
    """
    Build A_train, b_train, weights W.
    Weighting scheme (Step 1 / Step 6):
      2026 completed races → weight 3.0  (same regulations, most relevant)
      2025 rounds 15-24    → weight 1.5  (recent, slightly different regs)
      2025 rounds 1-14     → weight 0.5  (older, least relevant)
    """
    cutoff_2025 = int(n_2025 * 0.6)  # rounds 1..cutoff = low weight
    A_rows,b_rows,w_rows,r_labels=[],[],[],[]

    print(f"\n📡 Building training matrix")
    print(f"   2025 rounds   : 1–{n_2025}")
    print(f"   2026 completed: {season_2026_done} rounds")
    print("-"*52)

    loader = _load_real_round if USE_REAL else _synth_round

    # 2025 training data
    for rnd in range(1, n_2025+1):
        args=(2025,rnd,top_n) if USE_REAL else (2025,rnd,top_n)
        try:
            drv,teams_r,feats,pos=loader(*args)
            A_rows.append(feats); b_rows.extend(pos.tolist())
            wt=0.5 if rnd<=cutoff_2025 else 1.5
            w_rows.extend([wt]*len(drv))
            r_labels.extend([(2025,rnd)]*len(drv))
            src="FastF1" if USE_REAL else "synthetic"
            print(f"  2025 R{rnd:>2} ✅  {len(drv)} drivers  w={wt}  [{src}]")
        except Exception as e:
            print(f"  2025 R{rnd:>2} ⚠️  {e}")

    # 2026 completed races
    for rnd in range(1, season_2026_done+1):
        args=(2026,rnd,top_n) if USE_REAL else (2026,rnd,top_n)
        try:
            drv,teams_r,feats,pos=loader(*args)
            A_rows.append(feats); b_rows.extend(pos.tolist())
            w_rows.extend([3.0]*len(drv))
            r_labels.extend([(2026,rnd)]*len(drv))
            src="FastF1" if USE_REAL else "synthetic"
            print(f"  2026 R{rnd:>2} ✅  {len(drv)} drivers  w=3.0  [{src}] ← 2026")
        except Exception as e:
            print(f"  2026 R{rnd:>2} ⚠️  {e}")

    A_train=_np.vstack(A_rows)
    b_train=_np.array(b_rows,dtype=float)
    W_vec  =_np.array(w_rows,dtype=float)
    print("-"*52)
    print(f"  Training matrix : {A_train.shape[0]} rows × {A_train.shape[1]} features")
    print(f"  Weight range    : {W_vec.min():.1f} – {W_vec.max():.1f}")
    return A_train, b_train, W_vec, r_labels

print("✅ Data loader defined — Cell 3 ready")


---
## 🔢 Cell 4 — Linear Algebra Engine
All 7 pipeline steps. Step 6 uses Weighted Least Squares.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — LINEAR ALGEBRA ENGINE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import numpy as _np2

F1_RED='#E10600'; F1_GOLD='#FFD700'; F1_WHITE='#FFFFFF'

def normalise(M):
    mn,mx=M.min(0),M.max(0)
    rng=_np2.where(mx-mn==0,1,mx-mn)
    return (M-mn)/rng,mn,rng

def norm_apply(M,mn,rng): return (M-mn)/rng

# Step 2 — RREF
def rref(M,tol=1e-10):
    M=M.astype(float).copy(); rows,cols=M.shape; pr=0; pivots=[]
    for col in range(cols):
        if pr>=rows: break
        best=_np2.argmax(_np2.abs(M[pr:,col]))+pr
        if abs(M[best,col])<tol: continue
        M[[pr,best]]=M[[best,pr]]
        M[pr]=M[pr]/M[pr,col]
        for r in range(rows):
            if r!=pr: M[r]-=M[r,col]*M[pr]
        pivots.append(col); pr+=1
    return M,pivots

# Step 3 — SVD / spaces
def analyse_spaces(A):
    U,S,Vt=_np2.linalg.svd(A,full_matrices=True)
    rank=int(_np2.sum(S>1e-10))
    return U,S,Vt,rank

# Step 4 — Correlation
def correlation_matrix(A): return _np2.corrcoef(A.T)

# Step 5 — Gram-Schmidt / QR
def gram_schmidt_qr(M):
    Q,R=_np2.linalg.qr(M)
    return Q,R

# Step 6 — WEIGHTED Least Squares
# Formula: x̂ = (AᵀWA)⁻¹AᵀWb  where W = diag(weights)
# This is the standard WLS normal equation — same concept as OLS
# but each row of A is scaled by its weight before solving.
def weighted_least_squares(A, b, W_vec):
    """
    W_vec : 1-D array of per-row weights
    Constructs diagonal weight matrix W, then solves:
        x̂ = (AᵀWA)⁻¹AᵀWb
    via the equivalent scaled system:
        (√W · A) x = (√W · b)
    which numpy lstsq handles numerically stably.
    """
    sqrt_w = _np2.sqrt(W_vec)              # √W
    A_w    = A * sqrt_w[:, _np2.newaxis]   # scale rows of A
    b_w    = b * sqrt_w                    # scale b
    x_hat, _, _, _ = _np2.linalg.lstsq(A_w, b_w, rcond=None)
    b_hat  = A @ x_hat                    # predicted (unweighted)
    resid  = b - b_hat
    # weighted RMSE
    rmse   = float(_np2.sqrt(_np2.average(resid**2, weights=W_vec)))
    # orthogonality check: AᵀW·e should be ≈ 0
    orth_err = float(_np2.max(_np2.abs((A * W_vec[:,_np2.newaxis]).T @ resid)))
    return x_hat, b_hat, resid, rmse, orth_err

# Step 7 — Eigenanalysis of weighted AᵀWA
def eigenanalysis(A, W_vec):
    sqrt_w = _np2.sqrt(W_vec)
    A_w    = A * sqrt_w[:, _np2.newaxis]
    AtWA   = A_w.T @ A_w
    vals,vecs = _np2.linalg.eigh(AtWA)
    idx=_np2.argsort(vals)[::-1]
    vals,vecs=vals[idx],vecs[:,idx]
    var_pct=vals/vals.sum()*100
    return vals,vecs,var_pct

print("✅ LA engine ready — Cell 4 done")


---
## 📊 Cell 5 — Visualisation Suite

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — VISUALISATION SUITE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import numpy as _npv
import matplotlib.pyplot as _plt
import matplotlib.gridspec as _gs
F1_RED='#E10600'; F1_GOLD='#FFD700'; F1_WHITE='#FFFFFF'
TEAM_COLORS={"Red Bull Racing":"#3671C6","Mercedes":"#27F4D2","Ferrari":"#E8002D",
    "McLaren":"#FF8000","Aston Martin":"#229971","Williams":"#64C4FF",
    "Alpine":"#0093CC","Haas F1 Team":"#B6BABD","RB":"#6692FF",
    "Kick Sauber":"#52E252","Audi":"#FF5F00","Cadillac":"#B01E23"}
FEATURE_NAMES=["Quali Gap (s)","Tire Factor","Env. Index","Mean/FP2 Lap (s)","Pitstop (s)"]

def _tc(teams): return [TEAM_COLORS.get(t,F1_RED) for t in teams]

def plot_step1(A_test, drivers, teams, title=""):
    fig,ax=_plt.subplots(figsize=(10,5))
    fig.suptitle(f"Step 1 — Feature Matrix A  [{title}]",color=F1_WHITE,fontsize=12)
    im=ax.imshow(A_test,cmap="RdYlGn_r",aspect="auto",vmin=0,vmax=1)
    ax.set_xticks(range(len(FEATURE_NAMES)))
    ax.set_xticklabels([f.split(" ")[0] for f in FEATURE_NAMES],fontsize=9)
    ax.set_yticks(range(len(drivers))); ax.set_yticklabels(drivers,fontsize=9)
    ax.set_title("Normalised feature values (0=best, 1=worst in field)",color=F1_WHITE,fontsize=10)
    for i in range(len(drivers)):
        for j in range(A_test.shape[1]):
            ax.text(j,i,f"{A_test[i,j]:.2f}",ha="center",va="center",
                    fontsize=7,color="white" if A_test[i,j]>0.6 else "black")
    _plt.colorbar(im,ax=ax,shrink=0.8); _plt.tight_layout(); _plt.show()

def plot_step2(A_rref, pivot_cols, n_show=10):
    n_show=min(n_show,len(A_rref))
    fig,ax=_plt.subplots(figsize=(10,4))
    im=ax.imshow(A_rref[:n_show],cmap="coolwarm",aspect="auto",vmin=-0.5,vmax=1.2)
    ax.set_xticks(range(len(FEATURE_NAMES)))
    ax.set_xticklabels([f.split(" ")[0] for f in FEATURE_NAMES],fontsize=9)
    ax.set_yticks(range(n_show)); ax.set_yticklabels([f"Row {i+1}" for i in range(n_show)],fontsize=8)
    ax.set_title(f"Step 2 — RREF  (first {n_show} rows)  |  Rank = {len(pivot_cols)}",color=F1_WHITE)
    for pc in pivot_cols: ax.axvline(pc,color=F1_GOLD,linewidth=2,alpha=0.5)
    for i in range(n_show):
        for j in range(len(FEATURE_NAMES)):
            v=A_rref[i,j]
            if abs(v)>1e-8:
                ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=7,
                        color="white" if abs(v)>0.5 else "black")
    _plt.colorbar(im,ax=ax,shrink=0.6); _plt.tight_layout(); _plt.show()

def plot_step3(S):
    p=len(FEATURE_NAMES)
    fig,ax=_plt.subplots(figsize=(8,4))
    ax.bar(range(1,p+1),S[:p],color=F1_RED,edgecolor="#222",linewidth=0.5)
    ax.set_xlabel("Index"); ax.set_ylabel("Magnitude")
    ax.set_title("Step 3 — Singular Values of A",color=F1_WHITE)
    for i,s in enumerate(S[:p]): ax.text(i+1,s*1.04,f"{s:.3f}",ha="center",fontsize=9)
    _plt.tight_layout(); _plt.show()

def plot_step4(corr):
    p=len(FEATURE_NAMES)
    fig,ax=_plt.subplots(figsize=(7,6))
    im=ax.imshow(corr,cmap="RdBu",vmin=-1,vmax=1)
    labs=[f.split(" ")[0] for f in FEATURE_NAMES]
    ax.set_xticks(range(p)); ax.set_xticklabels(labs,fontsize=9)
    ax.set_yticks(range(p)); ax.set_yticklabels(labs,fontsize=9)
    ax.set_title("Step 4 — Feature Correlation Matrix",color=F1_WHITE)
    _plt.colorbar(im,ax=ax,shrink=0.8)
    for i in range(p):
        for j in range(p):
            ax.text(j,i,f"{corr[i,j]:.2f}",ha="center",va="center",
                    fontsize=9,color="white" if abs(corr[i,j])>0.5 else "#333")
    _plt.tight_layout(); _plt.show()

def plot_step5(A_n,Q,R):
    p=len(FEATURE_NAMES)
    fig,axes=_plt.subplots(1,3,figsize=(15,5))
    fig.suptitle("Step 5 — QR Decomposition  A = QR",color=F1_WHITE,fontsize=13)
    for ax,mat,t in zip(axes,[A_n[:p],Q[:p,:p],R],
        ["A (normalised)","Q (orthonormal)","R (upper triangular)"]):
        im=ax.imshow(mat,cmap="coolwarm",aspect="auto")
        ax.set_title(t,color=F1_WHITE,fontsize=10)
        _plt.colorbar(im,ax=ax,shrink=0.6)
    _plt.tight_layout(); _plt.show()

def plot_step6_train(A_train, b_train, b_hat_train, W_vec):
    """Show weighted fit quality on training data."""
    fig,axes=_plt.subplots(1,2,figsize=(13,5))
    fig.suptitle("Step 6 — Weighted Least Squares  (Training fit)",color=F1_WHITE,fontsize=12)
    ax=axes[0]
    ax.scatter(b_train,b_hat_train,c=W_vec,cmap="hot",s=60,alpha=0.7,edgecolors="none")
    lim=[0.5,b_train.max()+0.5]
    ax.plot(lim,lim,"--",color=F1_GOLD,linewidth=1.2,label="Perfect")
    ax.set_xlabel("Actual position"); ax.set_ylabel("Fitted position")
    ax.set_title("Actual vs Fitted  (colour = weight)",color=F1_WHITE); ax.legend()
    ax2=axes[1]
    resid=b_train-b_hat_train
    ax2.scatter(b_hat_train,resid,c=W_vec,cmap="hot",s=60,alpha=0.7,edgecolors="none")
    ax2.axhline(0,color=F1_GOLD,linewidth=1.2,linestyle="--")
    ax2.set_xlabel("Fitted position"); ax2.set_ylabel("Residual")
    ax2.set_title("Residual Plot  (e ⊥ Col(A) verified)",color=F1_WHITE)
    _plt.tight_layout(); _plt.show()

def plot_step7(vals,vecs,var_pct):
    p=len(FEATURE_NAMES)
    fig,axes=_plt.subplots(1,2,figsize=(13,5))
    fig.suptitle("Step 7 — Eigenanalysis of AᵀWA",color=F1_WHITE,fontsize=13)
    cum=_npv.cumsum(var_pct)
    ax=axes[0]
    ax.bar(range(1,p+1),var_pct,color=F1_RED,edgecolor="#222",linewidth=0.5)
    ax.plot(range(1,p+1),cum,"o-",color=F1_GOLD,linewidth=2,markersize=7,label="Cumulative")
    ax.axhline(80,color="#666",linestyle="--",linewidth=1,label="80% threshold")
    ax.set_xlabel("Principal Component"); ax.set_ylabel("Variance Explained (%)")
    ax.set_title("Scree Plot",color=F1_WHITE); ax.legend()
    for i,v in enumerate(var_pct): ax.text(i+1,v+0.8,f"{v:.1f}%",ha="center",fontsize=8)
    ax2=axes[1]
    im=ax2.imshow(vecs[:p],cmap="RdBu",vmin=-1,vmax=1,aspect="auto")
    ax2.set_xticks(range(p)); ax2.set_xticklabels(["PC"+str(i+1) for i in range(p)])
    ax2.set_yticks(range(p)); ax2.set_yticklabels([f.split(" ")[0] for f in FEATURE_NAMES],fontsize=9)
    ax2.set_title("Eigenvector Loading Matrix",color=F1_WHITE)
    _plt.colorbar(im,ax=ax2,shrink=0.8)
    for i in range(p):
        for j in range(p):
            ax2.text(j,i,f"{vecs[i,j]:.2f}",ha="center",va="center",
                     fontsize=8,color="white" if abs(vecs[i,j])>0.5 else "#333")
    _plt.tight_layout(); _plt.show()

def plot_single_podium(race_name, drivers, teams, b_pred, b_actual=None):
    """Plot prediction for one upcoming race."""
    ranked=sorted(zip(drivers,teams,b_pred),key=lambda x:x[2])
    n=len(ranked)
    fig,axes=_plt.subplots(1,2,figsize=(14,5))
    fig.suptitle(f"Step 8 — Predicted Podium: {race_name}",color=F1_WHITE,fontsize=13)

    # Bar chart
    ax=axes[0]
    all_d=[r[0] for r in ranked]
    all_s=[r[2] for r in ranked]
    all_t=[r[1] for r in ranked]
    scores_plot=[n+1-s for s in all_s]
    bc=[TEAM_COLORS.get(t,F1_RED) for t in all_t]
    ax.bar(range(n),scores_plot,color=bc,edgecolor="#111",linewidth=0.4,alpha=0.9)
    medals=["🥇","🥈","🥉"]
    for i in range(3):
        ax.text(i,scores_plot[i]+0.15,medals[i],ha="center",fontsize=14)
    if b_actual is not None:
        act_ranked=sorted(zip(drivers,b_actual),key=lambda x:x[1])
        act_scores=[n+1-s for _,s in act_ranked]
        ax.step(range(n),act_scores,where="mid",color=F1_GOLD,linewidth=2,label="Actual")
        ax.legend(fontsize=9)
    ax.set_xticks(range(n)); ax.set_xticklabels(all_d,rotation=45,fontsize=9)
    ax.set_ylabel("Predicted strength"); ax.set_title("Ranking",color=F1_WHITE)

    # Table
    ax2=axes[1]; ax2.axis("off")
    has_actual=b_actual is not None
    col_labels=["Pos","Driver","Team","Score","Actual"] if has_actual else ["Pos","Driver","Team","Score"]
    tdata=[]
    for i,(d,t,s) in enumerate(ranked):
        row=[f"P{i+1}",d,t[:12],f"{s:.2f}"]
        if has_actual:
            row.append(f"P{int(b_actual[drivers.index(d)])}")
        tdata.append(row)
    tbl=ax2.table(cellText=tdata,colLabels=col_labels,loc="center",cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1,1.5)
    for (r,c),cell in tbl.get_celld().items():
        cell.set_facecolor("#1a1a1a" if r>0 else F1_RED)
        cell.set_edgecolor("#333")
        cell.set_text_props(color="white")
        if 1<=r<=3: cell.set_facecolor("#2a1800")
    ax2.set_title(f"Full Rankings",color=F1_WHITE,fontsize=10,pad=10)
    _plt.tight_layout()
    fname=f"prediction_{race_name.replace(' ','_').replace('/','_')}.png"
    _plt.savefig(fname,dpi=150,bbox_inches="tight",facecolor="#0a0a0a")
    _plt.show()
    print(f"  💾 Saved {fname}")
    return ranked

def plot_three_race_summary(results_list):
    """Side-by-side podium comparison for 3 races."""
    n_races=len(results_list)
    fig,axes=_plt.subplots(1,n_races,figsize=(6*n_races,7))
    fig.suptitle("Step 8 — Predicted Podiums: Next "+str(n_races)+" Races",
                 color=F1_WHITE,fontsize=14,y=1.02)
    if n_races==1: axes=[axes]
    medals=["🥇","🥈","🥉"]
    for ax,(race_name,ranked) in zip(axes,results_list):
        top3=ranked[:3]
        for i,(d,t,s) in enumerate(top3):
            col=TEAM_COLORS.get(t,F1_RED)
            height=[3,2,1][i]
            ax.bar(i,height,color=col,edgecolor="#222",linewidth=0.5,width=0.6,alpha=0.9)
            ax.text(i,height+0.05,medals[i],ha="center",fontsize=16)
            ax.text(i,height/2,d,ha="center",va="center",fontsize=11,
                    color="white",fontweight="bold")
            ax.text(i,-0.15,t[:10],ha="center",fontsize=7,color="#aaa")
        ax.set_xlim(-0.6,2.6); ax.set_ylim(-0.4,4.2)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(race_name,color=F1_WHITE,fontsize=11,pad=8)
        ax.set_facecolor("#111")
    _plt.tight_layout()
    _plt.savefig("three_race_prediction.png",dpi=150,bbox_inches="tight",facecolor="#0a0a0a")
    _plt.show()
    print("  💾 Saved three_race_prediction.png")

print("✅ Visualisation suite ready — Cell 5 done")


---
## ⚙️ Cell 6 — Pipeline Runner
Wires everything together. Button triggers this.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — MASTER PIPELINE RUNNER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import numpy as _npr
F1_RED='#E10600'; F1_GOLD='#FFD700'; F1_WHITE='#FFFFFF'
TEAM_COLORS={"Red Bull Racing":"#3671C6","Mercedes":"#27F4D2","Ferrari":"#E8002D",
    "McLaren":"#FF8000","Aston Martin":"#229971","Williams":"#64C4FF",
    "Alpine":"#0093CC","Haas F1 Team":"#B6BABD","RB":"#6692FF",
    "Kick Sauber":"#52E252","Audi":"#FF5F00","Cadillac":"#B01E23"}
FEATURE_NAMES=["Quali Gap (s)","Tire Factor","Env. Index","Mean/FP2 Lap (s)","Pitstop (s)"]

# 2026 completed rounds and upcoming races
SEASON_2026_DONE = 3   # ← update this as more 2026 races complete
UPCOMING_RACES   = [   # (season, round, label)
    (2026, 4, "Miami GP"),
    (2026, 5, "Canadian GP"),
    (2026, 6, "Monaco GP"),
]

def run_full_pipeline(n_train_2025, top_n, predict_next=3):
    SEP="═"*60

    # ── 1. Build training matrix ───────────────────────────────────
    w_status.value="<span style=\'color:#FFD700\'>⏳ Step 1 — Loading & building matrix A...</span>"
    A_train_raw, b_train, W_vec, r_labels = build_training_matrix(
        n_train_2025, top_n, SEASON_2026_DONE)

    A_train, mn, rng = normalise(A_train_raw)
    n_feat=A_train.shape[1]

    print(f"\n{SEP}")
    print("  STEP 1 — MATRIX REPRESENTATION")
    print(SEP)
    print(f"  A_train shape    : {A_train.shape}")
    print(f"  Features         : {FEATURE_NAMES}")
    print(f"  Weight scheme    : 2026 data=3.0 | late 2025=1.5 | early 2025=0.5")
    print(f"  C4 note          : For future races FP2 pace replaces race lap time")
    print(f"  C5 note          : For future races historical team pit avg used")
    # Show test matrix for first upcoming race
    season0,rnd0,name0=UPCOMING_RACES[0]
    drv0,teams0,feat0_raw,_=(_load_future_quali if USE_REAL else _synth_round)(
        *((season0,rnd0,top_n) if not USE_REAL else (season0,rnd0,top_n)))
    feat0=norm_apply(feat0_raw,mn,rng)
    plot_step1(feat0,drv0,teams0,name0)

    # ── 2. RREF ────────────────────────────────────────────────────
    w_status.value="<span style=\'color:#FFD700\'>⏳ Step 2 — RREF / Gaussian Elimination...</span>"
    A_rref,pivot_cols=rref(A_train)
    rank_A=len(pivot_cols)
    print(f"\n{SEP}")
    print("  STEP 2 — MATRIX SIMPLIFICATION (RREF)")
    print(SEP)
    print(f"  Rank(A)     : {rank_A}")
    print(f"  Pivot cols  : {[FEATURE_NAMES[c] for c in pivot_cols]}")
    print(f"  Nullity     : {n_feat-rank_A}")
    print(f"  RREF first 5 rows:")
    hdr=f"  {'':>8}"+"".join(f"{f.split()[0]:>12}" for f in FEATURE_NAMES)
    print(hdr)
    for i in range(min(5,len(A_rref))):
        print(f"  Row {i+1:<4}"+"".join(f"{A_rref[i,j]:>12.4f}" for j in range(n_feat)))
    plot_step2(A_rref,pivot_cols)

    # ── 3. Space structure ─────────────────────────────────────────
    w_status.value="<span style=\'color:#FFD700\'>⏳ Step 3 — Column & Null Space...</span>"
    U,S,Vt,rank_svd=analyse_spaces(A_train)
    print(f"\n{SEP}")
    print("  STEP 3 — STRUCTURE OF THE SPACE")
    print(SEP)
    print(f"  Singular values  : {_npr.round(S[:n_feat],4)}")
    print(f"  dim(Col A)       : {rank_svd}")
    print(f"  dim(Null A)      : {n_feat-rank_svd}")
    nsp="Trivial — no anomalous drivers" if n_feat-rank_svd==0 else "Non-trivial"
    print(f"  Null Space       : {nsp}")
    plot_step3(S)

    # ── 4. Correlation / basis ─────────────────────────────────────
    w_status.value="<span style=\'color:#FFD700\'>⏳ Step 4 — Feature independence check...</span>"
    corr=correlation_matrix(A_train)
    print(f"\n{SEP}")
    print("  STEP 4 — REMOVE REDUNDANCY (BASIS SELECTION)")
    print(SEP)
    high=[(FEATURE_NAMES[i],FEATURE_NAMES[j],corr[i,j])
          for i in range(n_feat) for j in range(i+1,n_feat) if abs(corr[i,j])>0.85]
    if high:
        for a,b_,r in high: print(f"  ⚠️  High corr: {a} ↔ {b_}  r={r:.3f}")
    else: print("  ✅ No redundant features — all 5 columns kept in basis")
    print(f"  Basis dim : {rank_A}")
    plot_step4(corr)

    # ── 5. QR ──────────────────────────────────────────────────────
    w_status.value="<span style=\'color:#FFD700\'>⏳ Step 5 — Gram-Schmidt Orthogonalization...</span>"
    Q,R=gram_schmidt_qr(A_train)
    orth_err=float(_npr.max(_npr.abs(Q.T@Q-_npr.eye(n_feat))))
    print(f"\n{SEP}")
    print("  STEP 5 — ORTHOGONALIZATION (GRAM-SCHMIDT / QR)")
    print(SEP)
    print(f"  Q shape        : {Q.shape}")
    print(f"  R shape        : {R.shape}")
    print(f"  Max |QᵀQ − I| : {orth_err:.2e}  ← ≈ 0 ✅")
    print("  R (upper triangular):")
    hdr=f"  {'':>12}"+"".join(f"{f.split()[0]:>10}" for f in FEATURE_NAMES)
    print(hdr)
    for i in range(n_feat):
        print(f"  {FEATURE_NAMES[i][:12]:<12}"+"".join(f"{R[i,j]:>10.4f}" for j in range(n_feat)))
    plot_step5(A_train,Q,R)

    # ── 6. Weighted Least Squares ──────────────────────────────────
    w_status.value="<span style=\'color:#FFD700\'>⏳ Step 6 — Weighted Least Squares x̂=(AᵀWA)⁻¹AᵀWb...</span>"
    x_hat,b_hat_train,resid,rmse_train,orth=weighted_least_squares(A_train,b_train,W_vec)
    print(f"\n{SEP}")
    print("  STEP 6 — PROJECTION & WEIGHTED LEAST SQUARES")
    print(SEP)
    print("  Formula : x̂ = (AᵀWA)⁻¹AᵀWb")
    print("  W       = diag(w₁,...,wₙ)  — 2026 rows weighted 3×, late 2025 1.5×")
    print()
    print("  Coefficient vector x̂:")
    for f,c in zip(FEATURE_NAMES,x_hat):
        bar="█"*max(0,int(abs(c)*1.5))
        print(f"  {f:<22} {c:+.4f}  {bar}")
    print(f"\n  Weighted RMSE (train) : {rmse_train:.4f}")
    print(f"  Orthogonality ‖AᵀWe‖  : {orth:.2e}  ← ≈ 0 confirms e ⊥ Col(A) ✅")
    plot_step6_train(A_train,b_train,b_hat_train,W_vec)

    # ── 7. Eigenanalysis ───────────────────────────────────────────
    w_status.value="<span style=\'color:#FFD700\'>⏳ Step 7 — Eigenvalues of AᵀWA...</span>"
    eig_vals,eig_vecs,var_pct=eigenanalysis(A_train,W_vec)
    print(f"\n{SEP}")
    print("  STEP 7 — PATTERN DISCOVERY (EIGENVALUES OF AᵀWA)")
    print(SEP)
    print(f"  {'PC':<5}{'Eigenvalue':>13}{'Var%':>8}{'Cumul%':>9}  Top feature")
    print("  "+"-"*55)
    cum=0
    for i in range(n_feat):
        cum+=var_pct[i]
        dom=_npr.argmax(_npr.abs(eig_vecs[:,i]))
        print(f"  PC{i+1:<3}{eig_vals[i]:>13.4f}{var_pct[i]:>7.1f}%{cum:>8.1f}%  {FEATURE_NAMES[dom]}")
    plot_step7(eig_vals,eig_vecs,var_pct)

    # ── 8. Predict upcoming races ──────────────────────────────────
    w_status.value="<span style=\'color:#FFD700\'>⏳ Step 8 — Predicting next races...</span>"
    print(f"\n{SEP}")
    print("  STEP 8 — FINAL OUTPUT: NEXT RACE PREDICTIONS")
    print(SEP)

    races_to_predict=UPCOMING_RACES[:predict_next]
    all_ranked=[]

    for season,rnd,race_name in races_to_predict:
        print(f"\n  🏁 Predicting {race_name} (Round {rnd}/{season})...")
        # Load future race features
        if USE_REAL:
            drv,teams_t,feat_raw,b_act=_load_future_quali(season,rnd,top_n)
        else:
            drv,teams_t,feat_raw,b_act=_synth_round(season,rnd,top_n)

        feat_norm=norm_apply(feat_raw,mn,rng)
        b_pred=feat_norm@x_hat

        print(f"  {'Driver':<8}{'Team':<24}{'Predicted Score':>16}")
        print("  "+"-"*50)
        ranked_tmp=sorted(zip(drv,teams_t,b_pred),key=lambda x:x[2])
        medals=["🥇","🥈","🥉"]
        for i,(d,t,s) in enumerate(ranked_tmp):
            m=medals[i] if i<3 else f"P{i+1}"
            print(f"  {m} {d:<7} {t:<24} {s:>12.3f}")

        ranked=plot_single_podium(race_name,drv,teams_t,b_pred,b_act)
        all_ranked.append((race_name,ranked))

    if len(all_ranked)>1:
        plot_three_race_summary(all_ranked)

    # Final status
    podiums=" | ".join(
        f"{'🥇'+r[1][0][0]+' 🥈'+r[1][1][0]+' 🥉'+r[1][2][0]}" for r in all_ranked)
    w_status.value=(
        f"<span style=\'color:#27F4D2\'>"
        f"✅ Done!  {podiums}"
        f"  |  Weighted RMSE={rmse_train:.3f}"
        f"</span>")

print("✅ Pipeline runner ready")
print("\n🏁 ALL CELLS LOADED — configure Cell 2 and click  ▶  Run Pipeline")


---
## 📋 Pipeline Summary

| Step | LA Concept | Implementation | Output |
|------|-----------|---------------|--------|
| 1 | Matrix Representation | FastF1 API | A (n×5), b (n×1), W weights |
| 2 | RREF / Gaussian Elim | Custom RREF | rank(A), pivot columns |
| 3 | Column & Null Space | SVD | dim=5, null trivial |
| 4 | Basis Selection | Correlation | 5 independent features |
| 5 | Gram-Schmidt / QR | numpy QR | Q orthonormal, R upper triangular |
| 6 | **Weighted** Least Squares | x̂=(AᵀWA)⁻¹AᵀWb | Coefficients x̂ |
| 7 | Eigenvalues of AᵀWA | numpy eigh | Principal components |
| 8 | Final Output | A_test @ x̂ | Podium predictions for next 3 races |

**Key design decisions:**
- 2026 data weighted 3× over early 2025 to handle regulation changes
- New teams (Audi/Cadillac) use 2026-only data — no 2025 history
- Future race C4 uses FP2 pace; C5 uses historical team pit averages
- Weighted Least Squares is still Step 6 — same LA step, extended formula

---
*UE24MA241B — Linear Algebra and Its Applications · PES University*